# Mạng nơ-ron hồi tiếp

Trong mô-đun trước, chúng ta đã tìm hiểu về các biểu diễn ngữ nghĩa phong phú của văn bản. Kiến trúc mà chúng ta sử dụng nắm bắt ý nghĩa tổng hợp của các từ trong một câu, nhưng nó không xem xét đến **thứ tự** của các từ, bởi vì phép toán tổng hợp theo sau các embeddings đã loại bỏ thông tin này khỏi văn bản gốc. Vì các mô hình này không thể đại diện cho thứ tự từ, nên chúng không thể giải quyết các tác vụ phức tạp hoặc mơ hồ hơn như tạo văn bản hoặc trả lời câu hỏi.

Để nắm bắt ý nghĩa của một chuỗi văn bản, chúng ta sẽ sử dụng một kiến trúc mạng nơ-ron gọi là **mạng nơ-ron hồi tiếp**, hoặc RNN. Khi sử dụng RNN, ta truyền câu của mình qua mạng một token tại một thời điểm, và mạng sẽ tạo ra một **trạng thái**, rồi ta truyền trạng thái này lại cho mạng cùng với token tiếp theo.

![Hình ảnh minh họa một ví dụ về sự tạo ra mạng nơ-ron hồi tiếp.](../../../../../translated_images/vi/rnn.27f5c29c53d727b5.webp)

Với chuỗi token đầu vào $X_0,\dots,X_n$, RNN tạo ra chuỗi các khối mạng nơ-ron, và huấn luyện chuỗi này một cách toàn diện thông qua lan truyền ngược. Mỗi khối mạng lấy cặp $(X_i,S_i)$ làm đầu vào, và tạo ra $S_{i+1}$ làm kết quả. Trạng thái cuối cùng $S_n$ hoặc đầu ra $Y_n$ được đưa vào bộ phân loại tuyến tính để tạo ra kết quả. Tất cả các khối mạng sử dụng chung một bộ trọng số, và được huấn luyện toàn diện chỉ với một lần lan truyền ngược.

> Hình trên cho thấy mạng nơ-ron hồi tiếp ở dạng giải phóng cuộn (bên trái), và dạng biểu diễn hồi tiếp gọn hơn (bên phải). Điều quan trọng cần nhận ra là tất cả các ô RNN đều có cùng một bộ **trọng số có thể chia sẻ**.

Do các vectơ trạng thái $S_0,\dots,S_n$ được truyền qua mạng, RNN có khả năng học các phụ thuộc tuần tự giữa các từ. Ví dụ, khi từ *not* xuất hiện trong chuỗi, nó có thể học cách phủ định một số thành phần trong vectơ trạng thái.

Bên trong, mỗi ô RNN chứa hai ma trận trọng số: $W_H$ và $W_I$, cùng với độ lệch $b$. Ở mỗi bước RNN, với đầu vào $X_i$ và trạng thái đầu vào $S_i$, trạng thái đầu ra được tính là $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$, trong đó $f$ là hàm kích hoạt (thường là $\tanh$).

> Đối với các bài toán như tạo văn bản (mà chúng ta sẽ học ở đơn vị tiếp theo) hoặc dịch máy, ta cũng muốn lấy giá trị đầu ra ở mỗi bước RNN. Trong trường hợp này, còn có một ma trận khác là $W_O$, và đầu ra được tính là $Y_i=f(W_O\times S_i+b_O)$.

Hãy xem mạng nơ-ron hồi tiếp có thể giúp chúng ta phân loại tập dữ liệu tin tức của mình như thế nào.

> Đối với môi trường sandbox, chúng ta cần chạy ô lệnh sau để đảm bảo thư viện cần thiết được cài đặt, và dữ liệu được tải trước. Nếu bạn chạy trên máy cá nhân, bạn có thể bỏ qua ô lệnh tiếp theo.


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

Khi đào tạo các mô hình lớn, việc cấp phát bộ nhớ GPU có thể trở thành một vấn đề. Chúng ta cũng có thể cần thử nghiệm với các kích thước minibatch khác nhau, để dữ liệu vừa vặn trong bộ nhớ GPU của chúng ta, trong khi quá trình đào tạo vẫn đủ nhanh. Nếu bạn đang chạy mã này trên máy GPU của riêng bạn, bạn có thể thử điều chỉnh kích thước minibatch để tăng tốc quá trình đào tạo.

> **Lưu ý**: Một số phiên bản driver NVidia được biết là không giải phóng bộ nhớ sau khi đào tạo mô hình. Chúng tôi đang chạy nhiều ví dụ trong sổ tay này, và điều đó có thể gây ra việc hết bộ nhớ trong một số thiết lập nhất định, đặc biệt nếu bạn đang tự thực hiện các thử nghiệm trong cùng một sổ tay. Nếu bạn gặp phải một số lỗi lạ khi bắt đầu đào tạo mô hình, bạn có thể muốn khởi động lại kernel của sổ tay.


In [3]:
batch_size = 16
embed_size = 64

## Bộ phân loại RNN đơn giản

Trong trường hợp của một RNN đơn giản, mỗi đơn vị lặp lại là một mạng tuyến tính đơn giản, nhận vào một vector đầu vào và vector trạng thái, và tạo ra một vector trạng thái mới. Trong Keras, điều này có thể được biểu diễn bởi lớp `SimpleRNN`.

Mặc dù chúng ta có thể truyền các token mã hóa one-hot trực tiếp cho lớp RNN, nhưng đây không phải là một ý tưởng tốt vì chiều dữ liệu lớn của chúng. Do đó, chúng ta sẽ sử dụng một lớp embedding để giảm chiều dữ liệu của các vector từ, tiếp theo là một lớp RNN, và cuối cùng là một bộ phân loại `Dense`.

> **Lưu ý**: Trong những trường hợp chiều dữ liệu không quá cao, ví dụ như khi sử dụng phân đoạn token cấp ký tự, việc truyền các token mã hóa one-hot trực tiếp vào ô RNN có thể là hợp lý.


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **Lưu ý:** Chúng tôi sử dụng một lớp embedding chưa được huấn luyện ở đây cho đơn giản, nhưng để có kết quả tốt hơn, chúng ta có thể sử dụng một lớp embedding được huấn luyện trước bằng Word2Vec, như đã mô tả trong đơn vị trước. Đây sẽ là một bài tập hay cho bạn để điều chỉnh mã này làm việc với các embedding được huấn luyện trước.

Bây giờ hãy cùng huấn luyện RNN của chúng ta. RNN nói chung khá khó để huấn luyện, bởi vì khi các tế bào RNN được mở rộng dọc theo độ dài chuỗi, số lượng lớp tham gia vào quá trình lan truyền ngược sẽ khá lớn. Do đó, chúng ta cần chọn một tốc độ học nhỏ hơn, và huấn luyện mạng trên một tập dữ liệu lớn hơn để tạo ra kết quả tốt. Điều này có thể mất khá nhiều thời gian, vì vậy việc sử dụng GPU là ưu tiên.

Để tăng tốc, chúng ta sẽ chỉ huấn luyện mô hình RNN trên các tiêu đề tin tức, bỏ qua phần mô tả. Bạn có thể thử huấn luyện với phần mô tả và xem bạn có thể làm cho mô hình huấn luyện được hay không.


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> **Lưu ý** rằng độ chính xác có thể thấp hơn ở đây, vì chúng ta chỉ đang đào tạo trên các tiêu đề tin tức.


## Revisiting variable sequences 

Remember that the `TextVectorization` layer will automatically pad sequences of variable length in a minibatch with pad tokens. It turns out that those tokens also take part in training, and they can complicate convergence of the model.

There are several approaches we can take to minimize the amount of padding. One of them is to reorder the dataset by sequence length and group all sequences by size. This can be done using the `tf.data.experimental.bucket_by_sequence_length` function (see [documentation](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)). 

Another approach is to use **masking**. In Keras, some layers support additional input that shows which tokens should be taken into account when training. To incorporate masking into our model, we can either include a separate `Masking` layer ([docs](https://keras.io/api/layers/core_layers/masking/)), or we can specify the `mask_zero=True` parameter of our `Embedding` layer.

> **Note**: This training will take around 5 minutes to complete one epoch on the whole dataset. Feel free to interrupt training at any time if you run out of patience. What you can also do is limit the amount of data used for training, by adding `.take(...)` clause after `ds_train` and `ds_test` datasets.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


Bây giờ chúng ta đang sử dụng masking, chúng ta có thể huấn luyện mô hình trên toàn bộ bộ dữ liệu tiêu đề và mô tả.

> **Lưu ý**: Bạn có nhận thấy rằng chúng ta đã sử dụng bộ vectorizer được huấn luyện trên các tiêu đề tin tức, chứ không phải trên toàn bộ nội dung bài báo không? Có khả năng một số token sẽ bị bỏ qua, vì vậy tốt hơn là nên huấn luyện lại bộ vectorizer. Tuy nhiên, điều này có thể chỉ có tác động rất nhỏ, nên chúng ta sẽ giữ bộ vectorizer đã được huấn luyện trước đó để đơn giản.


## LSTM: Bộ nhớ ngắn hạn dài hạn

Một trong những vấn đề chính của RNN là **độ dốc biến mất**. RNN có thể rất dài, và có thể gặp khó khăn trong việc truyền độ dốc trở lại lớp đầu tiên của mạng trong quá trình lan truyền ngược. Khi điều này xảy ra, mạng không thể học được mối quan hệ giữa các token xa nhau. Một cách để tránh vấn đề này là giới thiệu **quản lý trạng thái rõ ràng** bằng cách sử dụng **cổng**. Hai kiến trúc phổ biến nhất có cổng là **bộ nhớ ngắn hạn dài hạn** (LSTM) và **đơn vị chuyển tiếp có cổng** (GRU). Chúng ta sẽ tìm hiểu về LSTM ở đây.

![Ảnh minh họa một ví dụ về tế bào bộ nhớ ngắn hạn dài hạn](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

Mạng LSTM được tổ chức tương tự như một RNN, nhưng có hai trạng thái được truyền từ lớp này sang lớp khác: trạng thái thực tế $c$, và vector ẩn $h$. Ở mỗi đơn vị, vector ẩn $h_{t-1}$ được kết hợp với đầu vào $x_t$, và cùng nhau kiểm soát trạng thái $c_t$ và đầu ra $h_{t}$ thông qua **các cổng**. Mỗi cổng có kích hoạt sigmoid (đầu ra trong khoảng $[0,1]$), có thể được coi như một mặt nạ bit khi nhân với vector trạng thái. LSTM có các cổng sau (từ trái sang phải trong ảnh trên):
* **cổng quên** quyết định các thành phần của vector $c_{t-1}$ cần được quên, và các thành phần cần được giữ lại.
* **cổng đầu vào** quyết định bao nhiêu thông tin từ vector đầu vào và vector ẩn trước đó nên được nhập vào vector trạng thái.
* **cổng đầu ra** lấy vector trạng thái mới và quyết định các thành phần nào sẽ được sử dụng để tạo ra vector ẩn mới $h_t$.

Các thành phần của trạng thái $c$ có thể được coi như các cờ có thể bật hoặc tắt. Ví dụ, khi gặp tên *Alice* trong chuỗi, chúng ta đoán rằng nó chỉ một người phụ nữ, và bật cờ trong trạng thái nói rằng chúng ta có một danh từ giống cái trong câu. Khi tiếp tục gặp các từ *and Tom*, chúng ta sẽ bật cờ thông báo rằng chúng ta có một danh từ số nhiều. Do đó, bằng cách thao tác trạng thái, chúng ta có thể theo dõi các thuộc tính ngữ pháp của câu.

> **Lưu ý**: Đây là một tài nguyên tuyệt vời để hiểu nội dung bên trong của LSTM: [Hiểu về mạng LSTM](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) của Christopher Olah.

Mặc dù cấu trúc bên trong của một tế bào LSTM có thể trông phức tạp, Keras ẩn triển khai này bên trong lớp `LSTM`, vì vậy việc duy nhất chúng ta cần làm trong ví dụ trên là thay thế lớp hồi quy:


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> **Lưu ý** rằng việc huấn luyện LSTM cũng khá chậm, và bạn có thể không thấy nhiều sự tăng chính xác ở đầu quá trình huấn luyện. Bạn có thể cần tiếp tục huấn luyện một thời gian để đạt được độ chính xác tốt.


## Mạng RNN hai chiều và đa lớp

Trong các ví dụ của chúng ta cho đến nay, các mạng hồi tiếp hoạt động từ đầu một chuỗi đến cuối chuỗi. Điều này có vẻ tự nhiên với chúng ta vì nó theo cùng chiều mà chúng ta đọc hoặc nghe giọng nói. Tuy nhiên, đối với các tình huống yêu cầu truy cập ngẫu nhiên vào chuỗi đầu vào, thì việc thực hiện tính toán hồi tiếp theo cả hai hướng sẽ hợp lý hơn. Các mạng RNN cho phép tính toán theo cả hai hướng gọi là RNN **hai chiều**, và chúng có thể được tạo ra bằng cách bao bọc lớp hồi tiếp với một lớp `Bidirectonal` đặc biệt.

> **Lưu ý**: Lớp `Bidirectional` tạo hai bản sao của lớp bên trong nó, và đặt thuộc tính `go_backwards` của một trong những bản sao đó thành `True`, khiến nó đi theo hướng ngược lại dọc theo chuỗi.

Các mạng hồi tiếp, dù một chiều hay hai chiều, đều nắm bắt các mẫu trong một chuỗi, và lưu trữ chúng vào các vectơ trạng thái hoặc trả về chúng như là đầu ra. Giống như với các mạng tích chập, chúng ta có thể xây dựng một lớp hồi tiếp khác theo sau lớp đầu tiên để nắm bắt các mẫu cấp cao hơn, xây dựng từ các mẫu cấp thấp hơn được trích xuất bởi lớp đầu tiên. Điều này dẫn đến khái niệm về một **mạng RNN đa lớp**, bao gồm hai hoặc nhiều mạng hồi tiếp, trong đó đầu ra của lớp trước được truyền làm đầu vào cho lớp tiếp theo.

![Hình ảnh hiển thị một mạng RNN đa lớp LSTM dài-ngắn hạn](../../../../../translated_images/vi/multi-layer-lstm.dd975e29bb2a59fe.webp)

*Hình ảnh từ [bài viết tuyệt vời này](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3) của Fernando López.*

Keras làm cho việc xây dựng các mạng này trở nên dễ dàng, vì bạn chỉ cần thêm nhiều lớp hồi tiếp vào mô hình. Đối với tất cả các lớp ngoại trừ lớp cuối cùng, chúng ta cần chỉ định tham số `return_sequences=True`, vì chúng ta cần lớp trả về tất cả các trạng thái trung gian chứ không chỉ trạng thái cuối cùng của phép tính hồi tiếp.

Hãy xây dựng một LSTM hai lớp hai chiều cho bài toán phân loại của chúng ta.

> **Lưu ý** mã này cũng sẽ mất khá nhiều thời gian để hoàn thành, nhưng nó cho chúng ta độ chính xác cao nhất mà chúng ta đã thấy cho đến nay. Vì vậy có thể đáng để chờ đợi và xem kết quả.


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## RNN cho các nhiệm vụ khác

Cho đến bây giờ, chúng ta đã tập trung vào việc sử dụng RNN để phân loại các chuỗi văn bản. Nhưng chúng có thể xử lý nhiều nhiệm vụ hơn nữa, chẳng hạn như tạo văn bản và dịch máy &mdash; chúng ta sẽ xem xét các nhiệm vụ đó trong phần tiếp theo.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố miễn trừ trách nhiệm**:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc sai sót. Tài liệu gốc bằng ngôn ngữ gốc nên được coi là nguồn tin chính thức. Đối với thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp bởi con người. Chúng tôi không chịu trách nhiệm về bất kỳ hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
